# 04 · The DL regime-MoE — a learned gate for the close regime

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** Each row is a real `resid_regime` cascade run on `linbest` with the **same d8@cs0.5
global** as ch. 03, swapping only the regime learner for an **interpretable mixture-of-experts**: a
learned soft-tree **gate** (the *automated* analog of the hand-found `hour` regime, ch. 01) routing
per-regime **NAM experts**, with an optional high-order MLP probe. `ctrl_ebm` reproduces the 0.12033
anchor so the MoE rows are comparable.

**Honest prior: low** — a tuned XGB regime already lost to the EBM (ch. 03). The likely payoff is the
**interpretable gate readout**, not the 4th-decimal QLIKE.

In [ ]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
print("setup ok")

---
## 1 · The new regime learner (folded)

The cascade is unchanged; only the regime learner is new (`_tree_factory("moe", cfg)`). Two pieces: the
**soft-tree gate** (read its hyperplane post-fit) and the **adapter** obeying the cascade `.fit/.predict`
contract (standardize, temporal val-tail early stopping, restore best weights).

In [ ]:
from src.models.regime_moe import SoftTreeGate, RegimeMoE
display(show_one(SoftTreeGate))
display(show_one(RegimeMoE.fit))

---
## 2 · Verify — the validation ladder (backfills from the overnight chain)

`submit_moe_chain.sh` → `results/moe_ladder/summary.csv` (rung, qlike_full, qlike_h16_19). Until it
lands the rows show `pending`. **`ctrl_ebm` must reproduce 0.12033.**

In [ ]:
CID = "xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim"
RUNGS = {
    "ctrl_ebm":   "EBM regime control (anchor 0.12033)",
    "moe_d0":     "depth-0 NAM — can any DL match the EBM?",
    "moe_d1":     "depth-1 gate (2 regimes) — does a learned partition beat doing-nothing (0.12081)?",
    "moe_d2":     "depth-2 gate (4 regimes, hierarchical)",
    "moe_d1diag": "depth-1 gate + high-order probe — is >=3-way the gap?",
}
p = REPO / "results" / "moe_ladder" / "summary.csv"
summary = pd.read_csv(p) if p.exists() and p.stat().st_size > 0 else None
rows = []
for lbl, desc in RUNGS.items():
    if summary is not None and lbl in set(summary.rung):
        r = summary[summary.rung == lbl].iloc[0]
        rows.append({"rung": lbl, "qlike_full": float(r.qlike_full),
                     "qlike_h16_19": float(r.qlike_h16_19), "source": "cluster-collect", "desc": desc})
    else:
        rows.append({"rung": lbl, "qlike_full": np.nan, "qlike_h16_19": np.nan, "source": "pending", "desc": desc})
tab = pd.DataFrame(rows); tab["d_vs_EBM"] = (tab.qlike_full - 0.12033).round(5)
display(tab[["rung", "qlike_full", "qlike_h16_19", "d_vs_EBM", "source", "desc"]])
ctrl = tab.loc[tab.rung == "ctrl_ebm", "qlike_full"]
if ctrl.notna().any():
    assert abs(float(ctrl.iloc[0]) - 0.12033) < 5e-4, "ctrl_ebm did NOT reproduce 0.12033 — path suspect"
    plot = tab.dropna(subset=["qlike_full"])
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(plot.rung, plot.d_vs_EBM, color=["#b06a6a" if d > 0 else "#2f6f4e" for d in plot.d_vs_EBM])
    ax.axvline(0, c="0.3", lw=1); ax.set_xlabel("ΔQLIKE vs EBM 0.12033 (neg = beats EBM)")
    ax.set_title("Does a learned regime gate beat the EBM regime stage?"); plt.tight_layout(); plt.show()
    print("PASS - ctrl_ebm reproduces 0.12033; MoE rows comparable.")
else:
    print("PENDING - overnight chain not yet collected; re-run after summary.csv lands.")

## 3 · Interpret — the ladder + the gate readout

1. **depth-0 NAM**: can any DL match the EBM? If not, the gate is moot (the EBM is already the right
   small-sample learner).
2. **depth-1/2 gate**: does a *learned* partition beat doing-nothing (0.12081) / the EBM (0.12033)? The
   one model-side lever the evidence hadn't foreclosed (learned localization × per-regime nonlinearity).
3. **diagnostic MLP**: is **≥3-way** structure the gap the EBM's pairwise form can't reach?

**The interpretable payoff — read the gate.** Even if QLIKE barely moves, the learned hyperplane answers
*where are the regimes?* `gate_readout.py` dumps the decision-node weights vs feature names: does the top
splitter recover `hour` (validating ch. 01's clock mechanism) or find a partition the clock smeared?

```bash
READOUT_DEPTH=1 $PY gate_readout.py xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim
```
*(gate readout + the depth-0/1/2/diag numbers fold in here once the chain completes.)*

The same architecture is what you point at the **auction-imbalance feed** once acquired — gate = regime,
experts = the new microstructure state. So it is not throwaway even if null on price-only features.